# SMOTE-Enhanced Stacking Ensemble Intrusion Detection System

## Phase 1: Data Loading and Initial Inspection

### Objectives
- Load all CICIDS2017 CSV files
- Verify the dataset structure
- Merge all CSV files into a single DataFrame
- Perform initial inspection of the dataset

**Dataset:** CICIDS2017 (MachineLearningCSV)

**Author:** Team

In [1]:
# ============================================================
# Import Required Libraries
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm

print("✅ Libraries imported successfully.")

✅ Libraries imported successfully.


In [3]:
# ============================================================
# Project Configuration
# ============================================================

PROJECT_DIR = Path(r"A:\Network-Intrusion-Detection-System")

DATASET_DIR = PROJECT_DIR / "dataset" / "raw" / "MachineLearningCVE"

print("Project Directory:")
print(PROJECT_DIR)

print("\nDataset Directory:")
print(DATASET_DIR)

Project Directory:
A:\Network-Intrusion-Detection-System

Dataset Directory:
A:\Network-Intrusion-Detection-System\dataset\raw\MachineLearningCVE


In [4]:
# ============================================================
# Locate All CSV Files
# ============================================================

csv_files = sorted(DATASET_DIR.glob("*.csv"))

print(f"Total CSV Files Found: {len(csv_files)}\n")

for file in csv_files:
    print(file.name)

Total CSV Files Found: 8

Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Friday-WorkingHours-Morning.pcap_ISCX.csv
Monday-WorkingHours.pcap_ISCX.csv
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Tuesday-WorkingHours.pcap_ISCX.csv
Wednesday-workingHours.pcap_ISCX.csv


In [5]:
# ============================================================
# Read One Sample CSV File
# ============================================================

sample_df = pd.read_csv(csv_files[0])

print("Dataset Shape:", sample_df.shape)

sample_df.head()

Dataset Shape: (225745, 79)


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,54865,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,55054,109,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,55055,52,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,46236,34,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,54863,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [6]:
# ============================================================
# Dataset Information
# ============================================================

sample_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 225745 entries, 0 to 225744
Data columns (total 79 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0    Destination Port             225745 non-null  int64  
 1    Flow Duration                225745 non-null  int64  
 2    Total Fwd Packets            225745 non-null  int64  
 3    Total Backward Packets       225745 non-null  int64  
 4   Total Length of Fwd Packets   225745 non-null  int64  
 5    Total Length of Bwd Packets  225745 non-null  int64  
 6    Fwd Packet Length Max        225745 non-null  int64  
 7    Fwd Packet Length Min        225745 non-null  int64  
 8    Fwd Packet Length Mean       225745 non-null  float64
 9    Fwd Packet Length Std        225745 non-null  float64
 10  Bwd Packet Length Max         225745 non-null  int64  
 11   Bwd Packet Length Min        225745 non-null  int64  
 12   Bwd Packet Length Mean       225745 non-nul

In [7]:
# ============================================================
# Column Names
# ============================================================

for i, column in enumerate(sample_df.columns, start=1):
    print(f"{i:02d}. {column}")

01.  Destination Port
02.  Flow Duration
03.  Total Fwd Packets
04.  Total Backward Packets
05. Total Length of Fwd Packets
06.  Total Length of Bwd Packets
07.  Fwd Packet Length Max
08.  Fwd Packet Length Min
09.  Fwd Packet Length Mean
10.  Fwd Packet Length Std
11. Bwd Packet Length Max
12.  Bwd Packet Length Min
13.  Bwd Packet Length Mean
14.  Bwd Packet Length Std
15. Flow Bytes/s
16.  Flow Packets/s
17.  Flow IAT Mean
18.  Flow IAT Std
19.  Flow IAT Max
20.  Flow IAT Min
21. Fwd IAT Total
22.  Fwd IAT Mean
23.  Fwd IAT Std
24.  Fwd IAT Max
25.  Fwd IAT Min
26. Bwd IAT Total
27.  Bwd IAT Mean
28.  Bwd IAT Std
29.  Bwd IAT Max
30.  Bwd IAT Min
31. Fwd PSH Flags
32.  Bwd PSH Flags
33.  Fwd URG Flags
34.  Bwd URG Flags
35.  Fwd Header Length
36.  Bwd Header Length
37. Fwd Packets/s
38.  Bwd Packets/s
39.  Min Packet Length
40.  Max Packet Length
41.  Packet Length Mean
42.  Packet Length Std
43.  Packet Length Variance
44. FIN Flag Count
45.  SYN Flag Count
46.  RST Flag Count
47. 

In [8]:
# ============================================================
# Target Column Distribution
# ============================================================

sample_df.iloc[:, -1].value_counts()

 Label
DDoS      128027
BENIGN     97718
Name: count, dtype: int64

In [9]:
# ============================================================
# Verify that all CSV files have the same schema
# ============================================================

reference_columns = None

for file in csv_files:
    df = pd.read_csv(file, nrows=5)  # Read only first 5 rows for validation

    if reference_columns is None:
        reference_columns = list(df.columns)

    if list(df.columns) != reference_columns:
        print(f"❌ Column mismatch found in: {file.name}")
    else:
        print(f"✅ {file.name} - Schema Verified")

✅ Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv - Schema Verified
✅ Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv - Schema Verified
✅ Friday-WorkingHours-Morning.pcap_ISCX.csv - Schema Verified
✅ Monday-WorkingHours.pcap_ISCX.csv - Schema Verified
✅ Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv - Schema Verified
✅ Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv - Schema Verified
✅ Tuesday-WorkingHours.pcap_ISCX.csv - Schema Verified
✅ Wednesday-workingHours.pcap_ISCX.csv - Schema Verified


In [10]:
# ============================================================
# Merge all CSV files into a single DataFrame
# ============================================================

dataframes = []

for file in tqdm(csv_files, desc="Loading CSV Files"):
    df = pd.read_csv(file)
    dataframes.append(df)

merged_df = pd.concat(dataframes, ignore_index=True)

print("\n✅ All CSV files merged successfully!")
print(f"Dataset Shape: {merged_df.shape}")

Loading CSV Files: 100%|██████████| 8/8 [00:15<00:00,  1.93s/it]



✅ All CSV files merged successfully!
Dataset Shape: (2830743, 79)


In [11]:
# ============================================================
# Initial Inspection of Merged Dataset
# ============================================================

print("Rows, Columns :", merged_df.shape)

print("\nFirst Five Rows")
display(merged_df.head())

print("\nLast Five Rows")
display(merged_df.tail())

Rows, Columns : (2830743, 79)

First Five Rows


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,54865,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,55054,109,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,55055,52,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,46236,34,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,54863,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN



Last Five Rows


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
2830738,53,32215,4,2,112,152,28,28,28.0,0.00000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2830739,53,324,2,2,84,362,42,42,42.0,0.00000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2830740,58030,82,2,1,31,6,31,0,15.5,21.92031,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2830741,53,1048635,6,2,192,256,32,32,32.0,0.00000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2830742,53,94939,4,2,188,226,47,47,47.0,0.00000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [12]:
# ============================================================
# Dataset Information
# ============================================================

merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2830743 entries, 0 to 2830742
Data columns (total 79 columns):
 #   Column                        Dtype  
---  ------                        -----  
 0    Destination Port             int64  
 1    Flow Duration                int64  
 2    Total Fwd Packets            int64  
 3    Total Backward Packets       int64  
 4   Total Length of Fwd Packets   int64  
 5    Total Length of Bwd Packets  int64  
 6    Fwd Packet Length Max        int64  
 7    Fwd Packet Length Min        int64  
 8    Fwd Packet Length Mean       float64
 9    Fwd Packet Length Std        float64
 10  Bwd Packet Length Max         int64  
 11   Bwd Packet Length Min        int64  
 12   Bwd Packet Length Mean       float64
 13   Bwd Packet Length Std        float64
 14  Flow Bytes/s                  float64
 15   Flow Packets/s               float64
 16   Flow IAT Mean                float64
 17   Flow IAT Std                 float64
 18   Flow IAT Max         

In [13]:
# ============================================================
# Missing Values
# ============================================================

missing = merged_df.isnull().sum()

missing = missing[missing > 0].sort_values(ascending=False)

print(missing)

Flow Bytes/s    1358
dtype: int64


In [14]:
# ============================================================
# Duplicate Records
# ============================================================

duplicates = merged_df.duplicated().sum()

print(f"Duplicate Rows : {duplicates}")

Duplicate Rows : 308381


In [15]:
# ============================================================
# Infinite Values
# ============================================================

import numpy as np

numeric_df = merged_df.select_dtypes(include=[np.number])

inf_values = np.isinf(numeric_df).sum().sum()

print(f"Infinite Values : {inf_values}")

Infinite Values : 4376


In [16]:
# ============================================================
# Attack Distribution
# ============================================================

label_column = merged_df.columns[-1]

merged_df[label_column].value_counts()

 Label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64

In [17]:
# ============================================================
# Save Merged Dataset
# ============================================================

output_path = PROJECT_DIR / "dataset" / "cleaned" / "merged_dataset.csv"

merged_df.to_csv(output_path, index=False)

print("✅ Merged dataset saved successfully!")
print(output_path)

✅ Merged dataset saved successfully!
A:\Network-Intrusion-Detection-System\dataset\cleaned\merged_dataset.csv


In [18]:
# ============================================================
# Verify All CSV Files Have the Same Schema
# ============================================================

reference_columns = None

for file in csv_files:
    df = pd.read_csv(file, nrows=5)

    if reference_columns is None:
        reference_columns = list(df.columns)

    if list(df.columns) == reference_columns:
        print(f"✅ {file.name}")
    else:
        print(f"❌ Schema Mismatch: {file.name}")

✅ Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
✅ Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
✅ Friday-WorkingHours-Morning.pcap_ISCX.csv
✅ Monday-WorkingHours.pcap_ISCX.csv
✅ Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
✅ Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
✅ Tuesday-WorkingHours.pcap_ISCX.csv
✅ Wednesday-workingHours.pcap_ISCX.csv


In [19]:
# ============================================================
# Merge All CSV Files
# ============================================================

from tqdm import tqdm

dataframes = []

for file in tqdm(csv_files, desc="Loading CSV Files"):
    df = pd.read_csv(file)
    dataframes.append(df)

merged_df = pd.concat(dataframes, ignore_index=True)

print("\n✅ Dataset Merged Successfully!")
print(f"Dataset Shape: {merged_df.shape}")

Loading CSV Files: 100%|██████████| 8/8 [00:27<00:00,  3.47s/it]



✅ Dataset Merged Successfully!
Dataset Shape: (2830743, 79)


In [20]:
merged_df.head()

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,54865,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,55054,109,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,55055,52,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,46236,34,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,54863,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [21]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2830743 entries, 0 to 2830742
Data columns (total 79 columns):
 #   Column                        Dtype  
---  ------                        -----  
 0    Destination Port             int64  
 1    Flow Duration                int64  
 2    Total Fwd Packets            int64  
 3    Total Backward Packets       int64  
 4   Total Length of Fwd Packets   int64  
 5    Total Length of Bwd Packets  int64  
 6    Fwd Packet Length Max        int64  
 7    Fwd Packet Length Min        int64  
 8    Fwd Packet Length Mean       float64
 9    Fwd Packet Length Std        float64
 10  Bwd Packet Length Max         int64  
 11   Bwd Packet Length Min        int64  
 12   Bwd Packet Length Mean       float64
 13   Bwd Packet Length Std        float64
 14  Flow Bytes/s                  float64
 15   Flow Packets/s               float64
 16   Flow IAT Mean                float64
 17   Flow IAT Std                 float64
 18   Flow IAT Max         

In [22]:
# ============================================================
# Missing Values Analysis
# ============================================================

missing_values = merged_df.isnull().sum()

missing_values = missing_values[missing_values > 0]

print("Total Columns with Missing Values:", len(missing_values))

if len(missing_values) > 0:
    display(missing_values.sort_values(ascending=False))
else:
    print("✅ No Missing Values Found")

Total Columns with Missing Values: 1


Flow Bytes/s    1358
dtype: int64

In [23]:
# ============================================================
# Duplicate Rows Analysis
# ============================================================

duplicate_rows = merged_df.duplicated().sum()

print(f"Duplicate Rows : {duplicate_rows}")

Duplicate Rows : 308381


In [24]:
# ============================================================
# Infinite Values Analysis
# ============================================================

import numpy as np

numeric_columns = merged_df.select_dtypes(include=[np.number])

infinite_values = np.isinf(numeric_columns).sum().sum()

print(f"Infinite Values : {infinite_values}")

Infinite Values : 4376


In [25]:
# ============================================================
# Attack Class Distribution
# ============================================================

label_column = merged_df.columns[-1]

class_distribution = merged_df[label_column].value_counts()

display(class_distribution)

 Label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64

In [26]:
# ============================================================
# Percentage Distribution
# ============================================================

percentage = (
    merged_df[label_column]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

display(percentage)

 Label
BENIGN                        80.30
DoS Hulk                       8.16
PortScan                       5.61
DDoS                           4.52
DoS GoldenEye                  0.36
FTP-Patator                    0.28
SSH-Patator                    0.21
DoS slowloris                  0.20
DoS Slowhttptest               0.19
Bot                            0.07
Web Attack � Brute Force       0.05
Web Attack � XSS               0.02
Infiltration                   0.00
Web Attack � Sql Injection     0.00
Heartbleed                     0.00
Name: proportion, dtype: float64

In [27]:
# ============================================================
# Remove Extra Spaces from Column Names
# ============================================================

merged_df.columns = merged_df.columns.str.strip()

print("✅ Column names cleaned.")

print("\nLast Five Columns:")

merged_df.columns[-5:]

✅ Column names cleaned.

Last Five Columns:


Index(['Idle Mean', 'Idle Std', 'Idle Max', 'Idle Min', 'Label'], dtype='object')

In [28]:
# ============================================================
# Verify Label Column
# ============================================================

print(merged_df.columns[-1])

Label
